# tuning_v1 — controlled CatBoost tuning
This notebook delegates to the repository `ml.tuning` CLI. Selection and threshold design use five nested, purged walk-forward validation folds built strictly inside train+validation; the test split is excluded. No deployment or live-trading behavior is changed.

In [ ]:
# Cell 1 — Mount persistent Drive storage.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Pinned repository and persistent paths.
REPO_URL = 'https://github.com/Qauntify/qauntify_webV1.git'
REPO_COMMIT = '354782f8300f5e5812854fc9dcde0c35beb2bb1c'
REPO_DIR = '/content/qauntify_webV1'
DRIVE_ROOT = '/content/drive/MyDrive/Quantify/training_v1_full_001'
DATASET_ROOT = f'{DRIVE_ROOT}/datasets/datasets/training_v1'
TUNING_DIR = f'{DRIVE_ROOT}/tuning/tuning_v1'

In [ ]:
# Cell 3 — Obtain the exact repository revision.
import pathlib, subprocess
if not pathlib.Path(REPO_DIR, '.git').is_dir():
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--detach', REPO_COMMIT], check=True)
assert pathlib.Path(DATASET_ROOT, 'training_manifest.json').is_file(), f'Missing frozen dataset: {DATASET_ROOT}'

In [ ]:
# Cell 4 — Install shared repository requirements.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements-training.txt'], check=True)

In [ ]:
# Cell 5 — Validate the frozen dataset and protected folds.
subprocess.run([sys.executable, '-m', 'ml.training.verify', '--config', 'ml/configs/catboost_v1.yaml', '--dataset-root', DATASET_ROOT], cwd=REPO_DIR, check=True)

In [ ]:
# Cell 6 — Full resumable tuning_v1 run. Do not add --smoke here.
subprocess.run([sys.executable, '-m', 'ml.tuning.cli', '--config', 'ml/configs/tuning_v1.yaml', '--dataset-root', DATASET_ROOT, '--output-dir', TUNING_DIR, '--resume'], cwd=REPO_DIR, check=True)

In [ ]:
# Cell 7 — Display the locked offline selection report.
print(pathlib.Path(TUNING_DIR, 'tuning_report.md').read_text())

## Resume after disconnect
Reconnect, rerun Cells 1–5 with unchanged paths, then rerun Cell 6. Completed fold jobs are skipped and CatBoost snapshots resume the current job.